In [325]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


This notebook documents the processing and harmonisation of the earthquake databases used for the record selection

In [326]:
from pathlib import Path
import pandas as pd
import numpy as np
from openquake.hazardlib.imt import AvgSA, SA, PGA, RSD595

from pickagm import eqdbases as eqdb
from pickagm import dfops 
from phd_project.config.config import load_config 
import phd_project.scripts.WP1_ground_motion_set.manage_flatfiles as mf
cfg = load_config()

In [327]:
# set some parameters
imts_to_keep = tuple(
    [AvgSA([0,3]).name, RSD595().name, PGA().name, SA(1.00).name])  # strings match


# ESM

In [328]:
esm_fp = cfg["raw_data"]["esm_flatfile"]
im_start_idx = 71
components = ["U", "V"]
desired_metadata = ["index", "database", "event_id", "event_time", "station_code", 
                    "location_code", "trt", "mag", "rjb", "vs30",
                    "max_usable_T", "component"]

In [329]:
esm_original = mf.load_esm_flatfile(esm_fp)  # loaded with all values as strings
metadata = eqdb.extract_metadata(esm_original, im_start_idx)
imdata = eqdb.extract_im_data(esm_original, im_start_idx)
metadata = mf.label_record_trts_esm(metadata) # add the trt type to each record
im_start_idx += 1         # one new column added


In [330]:
# rename the im columns
imdata = eqdb.rename_sa_columns_esm(imdata)
imdata.columns = pd.MultiIndex.from_tuples(
    [(c.split("_")[0], c.split("_")[1]) for c in imdata.columns])

# keep only U-V components of the records remove the rest
imdata = imdata[components]
imdata = imdata.astype(np.float64)

In [331]:
# stack arrays and add index and component columns
dfs = [pd.concat([metadata.assign(component=c), imdata[c]], axis=1) 
        for c in components]
df = pd.concat(dfs, axis=0)         # stack dfs vertically
df = df.reset_index() 
im_start_idx += 2           # two new columns added

In [332]:
# translate column names to OpenQuake IMT strings
columns = []
for c in df.columns:
    try:
        new_col = eqdb.esm_nonSA_im_translations[c]
    except KeyError:
        new_col = c
    columns.append(new_col)
df.columns = columns

In [333]:
# make PGA absolute
df["PGA"] = np.abs(df["PGA"])

# add the AvgSA([0, 3]) and AvgSA([0, 6]) columns
metadata = eqdb.extract_metadata(df, im_start_idx)
imdata = eqdb.extract_im_data(df, im_start_idx)
AvgSA_tags = ["AvgSA([0, 3])", "AvgSA([0, 6])"]
AvgSA_periods = [np.round(np.linspace(0, 3, 10), 3), 
                 np.round(np.linspace(0, 6, 10), 3)]

imdata = mf.add_AvgSA_columns2(imdata, AvgSA_tags, AvgSA_periods)    

# scale ims to desired units
imdata = mf.scale_cols(imdata, eqdb.esm_unit_conversions)

# drop unneeded ims
imdata = imdata[[c for c in imdata.columns if c.startswith(imts_to_keep)]]

# get the sa periods
sa_periods = mf.get_SA_periods_from_headers(df)


In [334]:
# organise the metadata and multiindex
metadata.loc[:, "max_usable_T"] = 0.8 / metadata[["U_hp", "V_hp"]].apply(pd.to_numeric).min(axis=1).copy()
metadata.loc[:, "mag"] = metadata["EMEC_Mw"].combine_first(metadata["Mw"]).copy()
metadata.loc[:, "rjb"] = metadata["JB_dist"].combine_first(metadata["epi_dist"]).copy()
metadata.loc[:, "vs30"] = metadata["vs30_m_sec"].combine_first(metadata["vs30_m_sec_WA"]).copy()
metadata.loc[:, "database"] = "ESM"
metadata = metadata[desired_metadata]
metadata.columns = pd.MultiIndex.from_tuples(
    [("metadata", c) for c in metadata.columns])
imdata.columns = pd.MultiIndex.from_tuples(
    [("ims", c) for c in imdata.columns])
df = pd.concat([metadata, imdata], axis=1)
esm = df.dropna(axis=0)
esm = esm.astype({
    ("metadata", "vs30"): np.float64,
    ("metadata", "rjb"): np.float64,
    ("metadata", "mag"): np.float64
})

esm_dtypes = esm.dtypes
print(esm.dtypes)

metadata  index              int64
          database          object
          event_id          object
          event_time        object
          station_code      object
          location_code     object
          trt               object
          mag              float64
          rjb              float64
          vs30             float64
          max_usable_T     float64
          component         object
ims       PGA              float64
          RSD595           float64
          SA(0.01)         float64
          SA(0.025)        float64
          SA(0.04)         float64
          SA(0.05)         float64
          SA(0.07)         float64
          SA(0.1)          float64
          SA(0.15)         float64
          SA(0.2)          float64
          SA(0.25)         float64
          SA(0.3)          float64
          SA(0.35)         float64
          SA(0.4)          float64
          SA(0.45)         float64
          SA(0.5)          float64
          SA(0.6)   

In [ ]:
# save the final dataframe
esm.to_csv(cfg["proc_data"]["gm_databases_for_selection"] / "esm_records.csv", 
          sep=",", index=False)

# NGA-Sub

In [249]:
# load the data
ngasub_meta_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_Metadata_SA_rotD50.csv"

ngasub_SA_H1_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H1.csv"
ngasub_SA_H2_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_SA_H2.csv"
ngasub_Duration_fp = cfg["raw_data"]["gm_flatfiles"] / "NGASub_duration_metrics_H1H2V.csv"

# components = ["U", "V"]
metadata_rename = {"NGAsubRSN":"index", "NGAsubEQID": "event_id", 
                   "NGAsubSSN": "station_code",
                   "Earthquake_Magnitude": "mag", 
                   "Intra_Inter_Flag": "trt",
                   "Rjb_km": "rjb", "Vs30_Selected_for_Analysis_m_s": "vs30",
                   "Longest_Usable_Period_for_PSa_Ave_Component_sec":"max_usable_T", 
                   }
trt_map = {0: "Subduction Interface",
           1: "Subduction Inslab"}

In [250]:
df_meta = pd.read_csv(ngasub_meta_fp, header=0, encoding="cp1252", 
                      dtype={"Interface_Event_FromHypocenterDepth": str,
                             "Instrument_Type": str,
                             "Geomatrix_Site_Code_1st_Letter": str})
df_meta = df_meta.iloc[:, 0:84]

In [251]:
# rename some columns and add some dummy metadata to match the columns in ESM
df_meta = df_meta.rename(columns=metadata_rename)
df_meta.loc[:, "database"] = "NGASub"
df_meta.loc[:, "event_time"] = "YYYY-MM-DD HH:MM:00"
df_meta.loc[:, "location_code"] = 0
df_meta.loc[:, "component"] = "H"
df_meta = df_meta[desired_metadata]
df_meta.columns = pd.MultiIndex.from_product([["metadata"], df_meta.columns])

In [252]:
# load the H1 spectral acceleration values
df_SA1 = pd.read_csv(ngasub_SA_H1_fp, header=0, encoding="cp1252", 
                      dtype={"Interface_Event_FromHypocenterDepth": str,
                             "Instrument_Type": str,
                             "Geomatrix_Site_Code_1st_Letter": str})
df_SA1 = df_SA1.iloc[:, 12:]
df_SA1 = df_SA1.drop(columns=["PGV_cm_sec", "PGD_cm"])

# rename the columns
new_sa_cols = ["PGA"] + [f"SA({c.split("pt")[0][1:]}.{c.split("pt")[1][:-1]})" for c in df_SA1.columns[1:]]
df_SA1.columns = new_sa_cols

# interpolate for the periods that we actually want
df_SA1 = df_SA1.replace(-999, np.nan)
interped_SA_1 = mf.interpolate_SA_values(df_SA1, sa_periods)

# calculate the average spectral acceleration
AvgSA_tags = ["AvgSA([0, 3])", "AvgSA([0, 6])"]
AvgSA_periods = [np.round(np.linspace(0, 3, 10), 3), 
                 np.round(np.linspace(0, 6, 10), 3)]

AvgSA_1 = mf.add_AvgSA_columns2(df_SA1, AvgSA_tags, AvgSA_periods)[AvgSA_tags] 


In [253]:
# load the H2 spectral acceleration values
df_SA2 = pd.read_csv(ngasub_SA_H2_fp, header=0, encoding="cp1252", 
                      dtype={"Interface_Event_FromHypocenterDepth": str,
                             "Instrument_Type": str,
                             "Geomatrix_Site_Code_1st_Letter": str})
df_SA2 = df_SA2.iloc[:, 12:]
df_SA2 = df_SA2.drop(columns=["PGV_cm_sec", "PGD_cm"])

# rename the columns
new_sa_cols = ["PGA"] + [f"SA({c.split("pt")[0][1:]}.{c.split("pt")[1][:-1]})" for c in df_SA2.columns[1:]]
df_SA2.columns = new_sa_cols

# interpolate for the periods that we actually want
df_SA2 = df_SA2.replace(-999, np.nan)
interped_SA_2 = mf.interpolate_SA_values(df_SA2, sa_periods)

# calculate the average spectral acceleration
AvgSA_2 = mf.add_AvgSA_columns2(df_SA2, AvgSA_tags, AvgSA_periods)[AvgSA_tags] 


In [254]:
# Load duration metrics
df_Dur = pd.read_csv(ngasub_Duration_fp, header=0, encoding="cp1252")
rsd595_1 = df_Dur["H1_Duration_0pt05_0pt95"]
rsd595_2 = df_Dur["H2_Duration_0pt05_0pt95"]   

In [336]:
# Stack 'em all together
H1_ims = pd.concat([rsd595_1.rename("RSD595"), AvgSA_1, df_SA1["PGA"], interped_SA_1], axis=1)
H1_ims.columns = pd.MultiIndex.from_product([["ims"], H1_ims.columns])
H2_ims = pd.concat([rsd595_2.rename("RSD595"), AvgSA_2, df_SA2["PGA"], interped_SA_2], axis=1)
H2_ims.columns = pd.MultiIndex.from_product([["ims"], H2_ims.columns])

H1_df = pd.concat([df_meta, H1_ims], axis=1)
H1_df[("metadata", "component")] = "H1"
H2_df = pd.concat([df_meta, H2_ims], axis=1)
H2_df[("metadata", "component")] = "H2"

NGAsub = pd.concat([H1_df, H2_df], axis=0)

# filter the TRT to only use Interface "0" and Inslab "1" events
mask = (NGAsub[("metadata", "trt")] == 0) | (NGAsub[("metadata", "trt")] == 1)
NGAsub = NGAsub.loc[mask, :]

# replace the trt integers with names
NGAsub[("metadata", "trt")] = NGAsub[("metadata", "trt")].replace(0, "Subduction Interface")
NGAsub[("metadata", "trt")] = NGAsub[("metadata", "trt")].replace(1, "Subduction Inslab")

# tweak the types to match esm df
NGAsub = NGAsub.astype(esm_dtypes)

# replace -999 with nan
NGAsub[("metadata", "vs30")] = NGAsub[("metadata", "vs30")].replace(-999, np.nan)
NGAsub[("metadata", "rjb")] = NGAsub[("metadata", "rjb")].replace(-999, np.nan)
NGAsub[("metadata", "max_usable_T")] = NGAsub[("metadata", "max_usable_T")].replace(-999, np.nan)
NGAsub = NGAsub.dropna(axis=0)

In [338]:
# check that all the columns are the same
all(NGAsub.columns.sort_values() == esm.columns.sort_values())

True

In [339]:
# save the final dataframe
NGAsub.to_csv(cfg["proc_data"]["gm_databases_for_selection"] / "NGAsub_records.csv", 
          sep=",", index=False)

# Final Database

In [341]:
final_db = pd.concat([esm, NGAsub], axis=0)

# save the final dataframe
NGAsub.to_csv(cfg["proc_data"]["gm_databases_for_selection"] / "ESM-NGAsub_combined.csv", 
          sep=",", index=False)